Training of a binary UNet model

In [5]:
import os
import random
import numpy as np
import rasterio
from skimage.transform import resize
from scipy.ndimage import rotate
import tensorflow as tf
from tensorflow.keras.utils import Sequence
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Conv2DTranspose, BatchNormalization, Concatenate
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import precision_score, recall_score, f1_score
import matplotlib.pyplot as plt

# -----------------------------
# Configuration
# -----------------------------
IMG_HEIGHT = 256
IMG_WIDTH = 256
BATCH_SIZE = 8
EPOCHS = 500

IMG_DIR = "../../data/tiled_data/images"
MASK_DIR = "../../data/tiled_data/masks_bin"  # Binary masks (0=background, 1=waste)
MODEL_DIR = "models_binary"
os.makedirs(MODEL_DIR, exist_ok=True)

# -----------------------------
# Custom Metrics
# -----------------------------
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

def iou_score(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(y_true, [-1])
    y_pred_f = tf.reshape(y_pred, [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

In [6]:
# -----------------------------
# Data Generator
# -----------------------------
class RasterDataGenerator(Sequence):
    def __init__(self, img_dir, mask_dir, img_files, mask_files, batch_size=BATCH_SIZE, shuffle=True, augment=True, **kwargs):
        super().__init__(**kwargs)
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.img_files = img_files
        self.mask_files = mask_files
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.augment = augment

        mask_set = set(mask_files)
        self.valid_indices = [i for i, f in enumerate(img_files) if f in mask_set]
        print(f"Generator initialized with {len(self.valid_indices)} matching pairs.")

        self.indices = np.array(self.valid_indices)
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        imgs, masks = [], []

        for i in batch_idx:
            file_name = self.img_files[i]
            with rasterio.open(os.path.join(self.img_dir, file_name)) as src:
                img = src.read([1,2,3]).transpose(1,2,0).astype("float32")
            with rasterio.open(os.path.join(self.mask_dir, file_name)) as src:
                mask = src.read(1).astype("float32")
                mask[mask > 0] = 1  # ensure binary

            img = resize(img, (IMG_HEIGHT, IMG_WIDTH), preserve_range=True)
            mask = resize(mask, (IMG_HEIGHT, IMG_WIDTH), order=0, preserve_range=True, anti_aliasing=False)

            if self.augment:
                if random.random() < 0.5:
                    img, mask = np.flipud(img), np.flipud(mask)
                if random.random() < 0.5:
                    angle = random.uniform(-15, 15)
                    img = rotate(img, angle, reshape=False, order=1, mode="reflect")
                    mask = rotate(mask, angle, reshape=False, order=0, mode="nearest")

            img = img / 255.0
            mask = np.expand_dims(mask, axis=-1)  # shape: HxWx1
            imgs.append(img)
            masks.append(mask)

        return np.array(imgs), np.array(masks)

In [7]:
# -----------------------------
# U-Net Model (Binary)
# -----------------------------
def build_unet_binary():
    inputs = Input((IMG_HEIGHT, IMG_WIDTH, 3))
    def conv_block(x, f):
        x = Conv2D(f, 3, padding="same", activation="relu")(x)
        x = BatchNormalization()(x)
        x = Conv2D(f, 3, padding="same", activation="relu")(x)
        return BatchNormalization()(x)
    s1 = conv_block(inputs, 64)
    p1 = MaxPooling2D((2,2))(s1)
    s2 = conv_block(p1, 128)
    p2 = MaxPooling2D((2,2))(s2)
    s3 = conv_block(p2, 256)
    p3 = MaxPooling2D((2,2))(s3)
    b1 = conv_block(p3, 512)
    d1 = Conv2DTranspose(256, 2, strides=2, padding="same")(b1)
    d1 = Concatenate()([d1, s3])
    d1 = conv_block(d1, 256)
    d2 = Conv2DTranspose(128, 2, strides=2, padding="same")(d1)
    d2 = Concatenate()([d2, s2])
    d2 = conv_block(d2, 128)
    d3 = Conv2DTranspose(64, 2, strides=2, padding="same")(d2)
    d3 = Concatenate()([d3, s1])
    d3 = conv_block(d3, 64)
    outputs = Conv2D(1, 1, activation="sigmoid")(d3)
    return Model(inputs, outputs)

In [9]:
# -----------------------------
# Train-Validation Split
# -----------------------------
all_img = sorted([f for f in os.listdir(IMG_DIR) if f.endswith(".tif")])
all_msk = sorted([f for f in os.listdir(MASK_DIR) if f.endswith(".tif")])
combined = list(zip(all_img, all_msk))
random.shuffle(combined)
split_idx = int(len(combined) * 0.8)
train_pairs = combined[:split_idx]
val_pairs = combined[split_idx:]
train_imgs, train_msks = zip(*train_pairs)
val_imgs, val_msks = zip(*val_pairs)

train_gen = RasterDataGenerator(IMG_DIR, MASK_DIR, train_imgs, train_msks)
val_gen = RasterDataGenerator(IMG_DIR, MASK_DIR, val_imgs, val_msks, shuffle=False, augment=False)

# -----------------------------
# Callbacks
# -----------------------------
early_stop = EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True)
checkpoint = ModelCheckpoint(filepath=os.path.join(MODEL_DIR, "best_unet_binary.keras"),
                             monitor='val_loss', save_best_only=True, verbose=1)
reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1)

Generator initialized with 820 matching pairs.
Generator initialized with 206 matching pairs.


In [10]:
# -----------------------------
# Compile & Train
# -----------------------------
model = build_unet_binary()
model.compile(optimizer="adam", loss="binary_crossentropy",
              metrics=["accuracy", dice_coef, iou_score])

history = model.fit(train_gen,
                    validation_data=val_gen,
                    epochs=EPOCHS,
                    callbacks=[early_stop, checkpoint, reduce_lr])

Epoch 1/500
103/103 ━━━━━━━━━━━━━━━━━━━━ 0s 4s/step - accuracy: 0.7374 - dice_coef: 0.1085 - iou_score: 0.0592 - loss: 0.6223
Epoch 1: val_loss improved from inf to 0.32141, saving model to models_binary\best_unet_binary.keras
103/103 ━━━━━━━━━━━━━━━━━━━━ 430s 4s/step - accuracy: 0.7384 - dice_coef: 0.1086 - iou_score: 0.0592 - loss: 0.6212 - val_accuracy: 0.9359 - val_dice_coef: 0.0880 - val_iou_score: 0.0470 - val_loss: 0.3214 - learning_rate: 0.0010
Epoch 2/500
103/103 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9498 - dice_coef: 0.1290 - iou_score: 0.0716 - loss: 0.2731
Epoch 2: val_loss improved from 0.32141 to 0.23923, saving model to models_binary\best_unet_binary.keras
103/103 ━━━━━━━━━━━━━━━━━━━━ 369s 4s/step - accuracy: 0.9498 - dice_coef: 0.1292 - iou_score: 0.0717 - loss: 0.2727 - val_accuracy: 0.9359 - val_dice_coef: 0.0492 - val_iou_score: 0.0254 - val_loss: 0.2392 - learning_rate: 0.0010
Epoch 3/500
103/103 ━━━━━━━━━━━━━━━━━━━━ 0s 3s/step - accuracy: 0.9491 - dice_coef

In [ ]:
import os
import random
import numpy as np
import rasterio
import tensorflow as tf
from tensorflow.keras.utils import Sequence
from tensorflow.keras.models import load_model
from skimage.transform import resize
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# --- 1. CONFIGURATION ---
IMG_HEIGHT = 256
IMG_WIDTH = 256
BATCH_SIZE = 8
IMG_DIR = "../../data/tiled_data/images"
MASK_DIR = "../../data/tiled_data/masks_bin"

# --- 2. DEFINE THE GENERATOR CLASS (Crucial to avoid NameError) ---
class RasterDataGenerator(Sequence):
    def __init__(self, img_dir, mask_dir, img_files, mask_files, batch_size=8, shuffle=True, augment=True):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.img_files = img_files
        self.mask_files = mask_files
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.augment = augment
        mask_set = set(mask_files)
        self.valid_indices = [i for i, f in enumerate(img_files) if f in mask_set]
        self.indices = np.array(self.valid_indices)
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.indices) / self.batch_size))

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        imgs, masks = [], []
        for i in batch_idx:
            file_name = self.img_files[i]
            with rasterio.open(os.path.join(self.img_dir, file_name)) as src:
                img = src.read([1,2,3]).transpose(1,2,0).astype("float32")
            with rasterio.open(os.path.join(self.mask_dir, file_name)) as src:
                mask = src.read(1).astype("float32")
                mask[mask > 0] = 1 
            
            img = resize(img, (IMG_HEIGHT, IMG_WIDTH), preserve_range=True)
            mask = resize(mask, (IMG_HEIGHT, IMG_WIDTH), order=0, preserve_range=True, anti_aliasing=False)
            
            img = img / 255.0  # Normalization
            mask = np.expand_dims(mask, axis=-1)
            imgs.append(img)
            masks.append(mask)
        return np.array(imgs), np.array(masks)

# --- 3. DEFINE CUSTOM METRICS (Required for load_model) ---
def dice_coef(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(tf.cast(y_true, tf.float32), [-1])
    y_pred_f = tf.reshape(tf.cast(y_pred, tf.float32), [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2. * intersection + smooth) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + smooth)

def iou_score(y_true, y_pred, smooth=1e-6):
    y_true_f = tf.reshape(tf.cast(y_true, tf.float32), [-1])
    y_pred_f = tf.reshape(tf.cast(y_pred, tf.float32), [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - intersection
    return (intersection + smooth) / (union + smooth)

# --- 4. DATA PREPARATION ---
all_img = sorted([f for f in os.listdir(IMG_DIR) if f.endswith(".tif")])
all_msk = sorted([f for f in os.listdir(MASK_DIR) if f.endswith(".tif")])
combined = list(zip(all_img, all_msk))
# Use same split as training (80/20)
split_idx = int(len(combined) * 0.8)
val_pairs = combined[split_idx:]
val_imgs, val_msks = zip(*val_pairs)

val_gen = RasterDataGenerator(IMG_DIR, MASK_DIR, val_imgs, val_msks, shuffle=False, augment=False)

# --- 5. EVALUATION ---
print("Loading model and running evaluation...")
model_path = "models_binary/best_unet_binary.keras"
eval_model = load_model(model_path, custom_objects={"dice_coef": dice_coef, "iou_score": iou_score})

y_true_all, y_pred_all = [], []
for i in range(len(val_gen)):
    x_batch, y_batch = val_gen[i]
    preds = eval_model.predict(x_batch, verbose=0)
    y_true_all.append(y_batch.ravel())
    y_pred_all.append((preds > 0.5).astype(np.uint8).ravel())

y_true_flat = np.concatenate(y_true_all)
y_pred_flat = np.concatenate(y_pred_all)

# --- 6. RESULTS ---
target_names = ["Background", "Waste"]
print("\n--- CLASSIFICATION REPORT (PRECISION, RECALL, F1) ---")
print(classification_report(y_true_flat, y_pred_flat, target_names=target_names))

cm = confusion_matrix(y_true_flat, y_pred_flat)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=target_names, yticklabels=target_names)
plt.title("Binary Confusion Matrix")
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.show()

NameError: name 'RasterDataGenerator' is not defined

In [10]:
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# 1. RE-DEFINE THE GENERATOR CLASS
# (This ensures 'val_gen' can be reconstructed)
from tensorflow.keras.utils import Sequence
import rasterio
from skimage.transform import resize

class RasterDataGenerator(Sequence):
    def __init__(self, img_dir, mask_dir, img_files, mask_files, batch_size=8, shuffle=False, augment=False):
        self.img_dir = img_dir
        self.mask_dir = mask_dir
        self.img_files = img_files
        self.mask_files = mask_files
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.img_files) / self.batch_size))

    def on_epoch_end(self):
        self.indices = np.arange(len(self.img_files))
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        imgs, masks = [], []
        for i in batch_idx:
            file_name = self.img_files[i]
            with rasterio.open(os.path.join(self.img_dir, file_name)) as src:
                img = src.read([1,2,3]).transpose(1,2,0).astype("float32") / 255.0
            with rasterio.open(os.path.join(self.mask_dir, file_name)) as src:
                mask = src.read(1).astype("float32")
                mask[mask > 0] = 1 
            
            img = resize(img, (256, 256), preserve_range=True)
            mask = resize(mask, (256, 256), order=0, preserve_range=True, anti_aliasing=False)
            imgs.append(img)
            masks.append(np.expand_dims(mask, axis=-1))
        return np.array(imgs), np.array(masks)

# 2. RE-PREPARE THE VALIDATION GENERATOR
IMG_DIR = "../../data/tiled_data/images"
MASK_DIR = "../../data/tiled_data/masks_bin"
all_img = sorted([f for f in os.listdir(IMG_DIR) if f.endswith(".tif")])
all_msk = sorted([f for f in os.listdir(MASK_DIR) if f.endswith(".tif")])
combined = list(zip(all_img, all_msk))

# Re-apply your 80/20 split
split_idx = int(len(combined) * 0.8)
val_pairs = combined[split_idx:]
val_imgs, val_msks = zip(*val_pairs)

val_gen = RasterDataGenerator(IMG_DIR, MASK_DIR, val_imgs, val_msks, shuffle=False)

# 3. LOAD MODEL & EVALUATE
# (Assuming dice_coef and iou_score are already defined in your notebook)
model = load_model('models_binary/best_unet_binary.keras', 
                   custom_objects={'dice_coef': dice_coef, 'iou_score': iou_score})

y_true_all, y_pred_all = [], []

print("Running Evaluation...")
for i in range(len(val_gen)):
    X, Y = val_gen[i]
    preds = model.predict(X, verbose=0)
    
    y_true_all.append(Y.ravel())
    y_pred_all.append((preds > 0.5).astype(np.uint8).ravel())

y_true_f = np.concatenate(y_true_all)
y_pred_f = np.concatenate(y_pred_all)

# 4. PRINT PRECISION, RECALL, F1
print("\n--- PER-CLASS EVALUATION ---")
print(classification_report(y_true_f, y_pred_f, target_names=["Background", "Waste"]))

Running Evaluation...

--- PER-CLASS EVALUATION ---
              precision    recall  f1-score   support

  Background       0.99      1.00      0.99  12845177
       Waste       0.89      0.80      0.84    655239

    accuracy                           0.99  13500416
   macro avg       0.94      0.90      0.92  13500416
weighted avg       0.99      0.99      0.99  13500416



In [12]:
# -----------------------------
# Save Final Model
# -----------------------------
save_path = os.path.join(MODEL_DIR, "unet_binary_final.h5")
model.save(save_path)
print(f"Final binary model saved at {save_path}")


Final binary model saved at models_binary\unet_binary_final.h5


In [13]:
# -----------------------------
# Plot Training History
# -----------------------------
PLOT_DIR = os.path.join(MODEL_DIR, "plots")
os.makedirs(PLOT_DIR, exist_ok=True)

# Loss
plt.figure(figsize=(8,6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title("Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(PLOT_DIR, "loss.png"))
plt.close()

# Accuracy
plt.figure(figsize=(8,6))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(PLOT_DIR, "accuracy.png"))
plt.close()

# Dice
plt.figure(figsize=(8,6))
plt.plot(history.history['dice_coef'], label='Training Dice')
plt.plot(history.history['val_dice_coef'], label='Validation Dice')
plt.title("Dice Coefficient")
plt.xlabel("Epoch")
plt.ylabel("Dice")
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(PLOT_DIR, "dice.png"))
plt.close()

# IoU
plt.figure(figsize=(8,6))
plt.plot(history.history['iou_score'], label='Training IoU')
plt.plot(history.history['val_iou_score'], label='Validation IoU')
plt.title("IoU Score")
plt.xlabel("Epoch")
plt.ylabel("IoU")
plt.legend()
plt.grid(True)
plt.savefig(os.path.join(PLOT_DIR, "iou.png"))
plt.close()

print(f"Training plots saved in {PLOT_DIR}")


Training plots saved in models_binary\plots
